# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors
This notebook provides a step-by-step guide for loading and exploring the FAIRˆ² Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/api/python/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL (see below), enabling programmatic, standards-based access to the dataset's rich structure, metadata, and data records.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`. This step ensures reproducibility and enables downstream programmatic processing.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")
print(f"Dataset identifier: {meta.identifier}")
print(f"Version: {meta.version}, Released: {meta.datePublished}")

## 2. Data Overview

Let's review the available Record Sets, their `@id` values, and fields. All further references to entities (Record Sets, Fields, Columns) will use their Croissant `@id` values for clarity and reproducibility.

> **Note:** `mlcroissant` exposes dataset structure via the metadata model; we will use it to discover available record sets and their fields.

In [ ]:
# List all Record Sets and selected Field schemas with their @id
record_sets = [rs for rs in meta.record_sets]
print("Available record sets:")
for i, rs in enumerate(record_sets):
    print(f"{i+1}. @id: {rs.id}, name: {rs.name}")
    if hasattr(rs, 'fields') and rs.fields:
        print("    Fields:")
        for field in rs.fields:
            print(f"        - @id: {field.id}, name: {getattr(field, 'name', '')}, dataType: {getattr(field, 'data_type', '')}")
    print()

# If only one record set exists, pick it for downstream usage
if len(record_sets) == 1:
    main_rs = record_sets[0]
    main_rs_id = main_rs.id
else:
    # If more than one, user can choose; for notebook, pick the first
    main_rs = record_sets[0]
    main_rs_id = main_rs.id

print(f"Main record set selected: {main_rs_id}")

## 3. Data Extraction

Now, let's load the data from the selected record set directly into a pandas DataFrame. We will extract the records using the record set's Croissant `@id`.

> **Tip:** To work with multiple record sets, modify the list of `@id` values as needed.

In [ ]:
# Prepare list of record set @ids for extraction
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    print(f"Loading records for Record Set: {rs_id}")
    records_iter = dataset.records(record_set=rs_id)
    records = list(records_iter)
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

print(f"\nColumns loaded for main record set ({main_rs_id}): ")
print(dataframes[main_rs_id].columns.tolist())
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)

Let's apply common data processing steps:
 - Filtering records based on numeric field values
 - Normalizing fields
 - Grouping/categorizing records

For demonstration, we'll select a numeric field, filter on its value, normalize it, and (if available) group by a categorical attribute. All field access uses `@id`.

In [ ]:
# Choose a numeric field for EDA. Display available numeric fields as a reference.
rs_fields = main_rs.fields
numeric_field = None
for field in rs_fields:
    # Croissant types for numbers: 'schema:Number', 'schema:Integer', 'schema:Float' (sometimes simplified)
    dt = getattr(field, 'data_type', '') or getattr(field, 'dataType', '')
    if dt and ("Number" in dt or "Integer" in dt or "Float" in dt or dt == 'number' or dt == 'integer' or dt == 'float'):
        numeric_field = field.id
        print(f"Using numeric field: {field.name} (@id: {field.id})")
        break

if not numeric_field:
    print("No numeric field detected in record set. Please inspect df columns.")
else:
    df = dataframes[main_rs_id]

    # Filter (if possible - if field has non-null values and is 'number' type)
    try:
        threshold = df[numeric_field].dropna().astype(float).quantile(0.5)  # median as threshold
        filtered_df = df[df[numeric_field].astype(float) > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize the field
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()
        ) / filtered_df[numeric_field].astype(float).std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a categorical field (first string/categorical found)
        group_field = None
        for field in rs_fields:
            dt = getattr(field, 'data_type', '') or getattr(field, 'dataType', '')
            if dt and ("Text" in dt or dt == 'string' or dt == 'text') and field.id != numeric_field:
                group_field = field.id
                print(f"Grouping by categorical field: {field.name} (@id: {field.id})")
                break

        if group_field and group_field in filtered_df.columns:
            grouped_df = (
                filtered_df.groupby(group_field)[numeric_field].mean().to_frame("mean_")+\
                filtered_df.groupby(group_field)[numeric_field].count().to_frame("count_")
            )
            print(f"Grouped statistics by {group_field}:")
            print(grouped_df.head())
    except Exception as e:
        print(f"Could not perform numeric filter or normalization: {e}")

## 5. Visualization

Let us visualize the distribution of the chosen numeric variable, and if categories exist, display a count plot by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if numeric_field and numeric_field in df.columns and df[numeric_field].notnull().sum() > 0:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].astype(float), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field and group_field in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=df[group_field], y=df[numeric_field].astype(float))
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=30)
        plt.show()
else:
    print("Numeric field not available for visualization.")

## 6. Conclusion

In this notebook, we demonstrated how to programmatically load and explore a Croissant-compliant clinical dataset using the `mlcroissant` library:
- Loaded dataset metadata and identified record sets, fields, and their `@id` values.
- Loaded records into a pandas DataFrame for inspection and analysis.
- Performed basic EDA: numeric filtering, normalization, and grouping.
- Visualized data distributions and relationships.

This FAIR^2 tabular dataset supports reproducible research into clinicopathological predictors and biomarkers in cancer survivors with second primary colorectal cancer. All processing referenced dataset elements strictly by their Croissant `@id` identifiers, enabling reliable and standards-driven workflows.